# 🚀 Project 1: 智能客服与舆情分析 (Smart Customer Service Analytics)

**背景 (Scenario)**:
你是某电商平台 (e-Commerce) 的数据分析师。最近客服团队反馈 "退款" 相关的投诉激增，但缺乏量化证据。
产品经理 (PM) 丢给你一份 `customer_feedback.csv`，希望能挖掘出用户不满的 **根本原因 (Root Cause)**。

**你的任务**:
1.  **数据清洗**: 规范化文本 (大小写、去标点)。
2.  **标签提取**: 用 `apply` + `lambda` 自动给每条评论打标签 (如 "物流慢", "质量差")。
3.  **量化分析**: 用 `groupby` 计算不同类型的投诉率和满意度。
4.  **抓取重点**: 用 `query` 筛选出需要紧急处理的 "高危用户"。

---

In [1]:
import pandas as pd
import numpy as np

# 🛠️ 0. 生成模拟数据 (无需修改，直接运行)
# -----------------------------------------------------
np.random.seed(42)
data = {
    'order_id': [f'ORD-{i}' for i in range(1001, 1021)],
    'user_comment': [
        "Delivery was super slow! I waited for 10 days.",
        "Great product, loved the quality.",
        "REFUND PLEASE! The item is broken.",
        "Customer service was rude and unhelpful.",
        "Fast shipping, but the package was damaged.",
        "Not what I expected. Want to return it.",
        "Amazing experience, will buy again!",
        "Terrible. The app crashed when I paid.",
        "Where is my refund? It's been a week.",
        "Quality is okay, but expensive.",
        "Shipping fee is too high.",
        "I love it! Best purchase ever.",
        "The size is wrong. I ordered M but got S.",
        "Totally scam. Do not buy.",
        "Neutral experience.",
        "Can I change the delivery address?",
        "System error during checkout.",
        "Highly recommended!",
        "Battery life is very short.",
        "Useless support. Nobody answered my call."
    ],
    'rating': [1, 5, 1, 1, 3, 2, 5, 1, 1, 3, 2, 5, 2, 1, 3, 4, 1, 5, 2, 1],
    'platform': np.random.choice(['App', 'Web'], 20)
}
df = pd.DataFrame(data)
df.head()

,order_id,user_comment,rating,platform
0,ORD-1001,Delivery was super slow! I waited for 10 days.,1,App
1,ORD-1002,"Great product, loved the quality.",5,Web
2,ORD-1003,REFUND PLEASE! The item is broken.,1,App
3,ORD-1004,Customer service was rude and unhelpful.,1,App
4,ORD-1005,"Fast shipping, but the package was damaged.",3,App


### Step 1: 文本预处理 (Text Cleaning)
**目标**: 所有的 NLP 分析都始于清洗。请把 `user_comment` 列统一成小写 (lowercase)。
**技巧**: `.str.lower()`

In [3]:
# Your solution here
df['user_comment'] = df['user_comment'].str.lower()

### Step 2: 关键词提取 (Feature Engineering)
**目标**: PM 想知道多少差评是因为 "退款 (Refund)" 难导致的。
请创建一个新列 `is_refund_issue` (Boolean): 如果评论里包含 "refund" 或 "return"，则为 True，否则 False。
**技巧**: `.str.contains('refund|return', case=False)`

In [4]:
# Your solution here
df['is_refund_issus'] = df['user_comment'].str.contains('refund|return', case=False)
df

,order_id,user_comment,rating,platform,is_refund_issus
0,ORD-1001,delivery was super slow! i waited for 10 days.,1,App,False
1,ORD-1002,"great product, loved the quality.",5,Web,False
2,ORD-1003,refund please! the item is broken.,1,App,True
3,ORD-1004,customer service was rude and unhelpful.,1,App,False
4,ORD-1005,"fast shipping, but the package was damaged.",3,App,False
5,ORD-1006,not what i expected. want to return it.,2,Web,True
6,ORD-1007,"amazing experience, will buy again!",5,App,False
7,ORD-1008,terrible. the app crashed when i paid.,1,App,False
8,ORD-1009,where is my refund? it's been a week.,1,App,True
9,ORD-1010,"quality is okay, but expensive.",3,Web,False


### Step 3: 情绪打标 (Lambda Logic)
**目标**: 根据 `rating` 进行简单的自动分级。
规则:
- 5分 -> 'Promoter' (推荐者)
- 3-4分 -> 'Passive' (中立者)
- 1-2分 -> 'Detractor' (贬损者/高危)

请用 `apply` + 自定义函数生成 `nps_segment` 列。

In [5]:
# Your solution here
rating_map = {
    1: 'detractor',
    2: 'detractor',
    3: 'passive',
    4: 'passive',
    5: 'promoter'
}

df['nps_segment'] = df['rating'].map(rating_map)
df

,order_id,user_comment,rating,platform,is_refund_issus,nps_segment
0,ORD-1001,delivery was super slow! i waited for 10 days.,1,App,False,detractor
1,ORD-1002,"great product, loved the quality.",5,Web,False,promoter
2,ORD-1003,refund please! the item is broken.,1,App,True,detractor
3,ORD-1004,customer service was rude and unhelpful.,1,App,False,detractor
4,ORD-1005,"fast shipping, but the package was damaged.",3,App,False,passive
5,ORD-1006,not what i expected. want to return it.,2,Web,True,detractor
6,ORD-1007,"amazing experience, will buy again!",5,App,False,promoter
7,ORD-1008,terrible. the app crashed when i paid.,1,App,False,detractor
8,ORD-1009,where is my refund? it's been a week.,1,App,True,detractor
9,ORD-1010,"quality is okay, but expensive.",3,Web,False,passive


### Step 4: 根因分析 (Aggregation)
**目标**: 统计不同渠道 (platform) 的 "退款问题占比" 和 "平均评分"。
**预期结果**: 一个 Summary DataFrame，包含 `mean_rating` 和 `refund_rate`。
**技巧**: `groupby('platform').agg(...)`

In [10]:
# Your solution here
summary_df = df.groupby('platform').agg(
    mean_rating=('rating','mean'),
    refund_rate=('is_refund_issus','mean'),
    refund_issus=('is_refund_issus','sum'),
    total_issue=('order_id','nunique')
)
summary_df

,mean_rating,refund_rate,refund_issus,total_issue
platform,,,,
App,2.153846,0.153846,2,13
Web,3.000000,0.142857,1,7


### Step 5: 高危预警 (Final Filter)
**目标**: 找出所有 **(评分 < 2) 且 (涉及 'service/support' 问题)** 的订单，导出给客服主管处理。
**技巧**: `query()` 或 Boolean Indexing + `str.contains()`
**注意**: `service/support` 指的是评论里包含 'service' 或 'support'。

In [11]:
# Your solution here
df.loc[(df['rating']<2)&(df['user_comment'].str.contains(r'service|support',regex=True,case=True))]

,order_id,user_comment,rating,platform,is_refund_issus,nps_segment
3,ORD-1004,customer service was rude and unhelpful.,1,App,False,detractor
19,ORD-1020,useless support. nobody answered my call.,1,App,False,detractor
